# Mixed layer heat budget analysis of ACCESS-OM2 runs

This notebook contains code to analyse the mixed layer temperature budget in ACCESS-OM2 for a given time period or event of interest.

This notebook relies on pre-computed grouped budget quantities as computed by the budget processing scripts in this repository (e.g. `Process_Online_Budget.ipynb`).

The theory behind this budget and the diagnostics used to analyse it are summarized in `Theory_and_Diagnostics.ipynb`, and in a paper in preparation.

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
import string
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data

### Define paths, region and time period to load data for

In [ ]:
base = '/scratch/e14/rmh561/access-om2/archive/025deg_jra55_iaf_cycle6_online_mlt/'          # Location of simulation output
pp_diags_folder = base + 'post_processed_diags/'                                                  # Temp folder where post-processed diagnostics (pre-processed budgets) are contained

# Choose year to consider:
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors
base2 = base + 'output%03d/ocean/' % output

# Climatology:
clim_str = 'output336-365' # 336-365 = 1989-2018
clim_label = '1989-2018'

# Region to load:
#reg = [-100, 20, 0, 70] # North Atlantic
#reg = [-100, -40, 0, 30] # North Atlantic
#reg = [-230, -190, -50, -10] # EAC
reg = [135-360,175-360, -60, -20] # SE Aus (Kajtar et al. 2022)
#reg = [None,None,None,None] # Globe
#reg = [-270, -70, -60, 60] # Pacific
#reg = [-270, -210, -20, 20] # Maritime continent
#reg = [-270, -180, -45, 0] # Australia

# Time period to load:
#times = slice('2019-01-01','2019-01-31')#lice(None,None)
#times_snap = slice('2019-01-01','2019-02-01') # Note; this must be 1 more than times.
times = slice('2017-09-01',None)
times_snap = slice('2017-09-01',None) # Note; this must be 1 more than times.
#times = slice(None,None)
#times_snap = slice(None,None) # Note; this must be 1 more than times.

# Chunks to use:
chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

# Set constants:
rho0 = 1035.
Cp = 3992.10322329649

# Whether to load/do hat-averaging:
do_hatavg = True

### Load standard daily and grid data

In [ ]:
# Grid file:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)

# Standard daily variables:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

### Load pre-computed MLT budget terms
These files contain pre-computed grouped MLT budget terms. Computed using `daily_online_mlt_budget_year.sub` and `spawn_daily_online_mlt_budget_year.py` and the functions defined below.

The resulting dataset `mlt_budget_stavg_daily` will contain x*y*t arrays with the above budget groups (e.g. advection, surface forcing etc.) as well as:
- fixedh\_tendency: The tendency term of mixed layer temperature with a fixed ML depth (i.e. the sum of all the RHS terms listed above, including the correction terms).
- residual: The residual, fixedh\_tendency minus all the RHS terms. This should be exactly zero. If it isn't, you're missing terms in the source MOM5 budget or something has gone wrong.
- mlt\_tendency: The tendency of the mixed layer temperature computed (in this case) from snapshots of the mixed layer temperature (the diagnostic temp\_in\_mld) at the start and end of the day.
- entrainment: The entrainment term, computed by residual mlt\_tendency - fixedh\_tendency (with the later here equal to the sum of the RHS terms, so the mlt budget closes)

All terms have units of degC/second.

We also optionally load a climatology of the same budget files. Note: to compute these, just use `ncea *.nc output.nc` from the command line, including all the years desired.

In [ ]:
# Pre-computed standard average online daily
mlt_budget_stavg_daily = xr.open_dataset(pp_diags_folder + 'mlt_budget_online_stavg/mlt_budget_stavg_daily_online_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times)

# Pre-computed standard average online climatology (1989-2018, outputs 336-365):
mlt_budget_stavg_clim = xr.open_dataset(pp_diags_folder + 'mlt_budget_online_stavg/mlt_budget_stavg_daily_online_' + clim_str + '_monthly_mean.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

In [ ]:
if do_hatavg:
    # Pre-computed falling average online daily
    mlt_budget_falavg_daily = xr.open_dataset(pp_diags_folder + 'mlt_budget_online_falavg/mlt_budget_falavg_daily_online_output%03d.nc' % output).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).sel(time=times)
    
    # Pre-computed standard average online climatology (1989-2018, outputs 336-365):
    mlt_budget_falavg_clim = xr.open_dataset(pp_diags_folder + 'mlt_budget_online_falavg/mlt_budget_falavg_daily_online_' + clim_str + '_monthly_mean.ncea.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

### Load climatologies of selected standard variables

In [ ]:
# Standard average daily budget diagnostics:
ds_clim = xr.open_dataset(pp_diags_folder + 'ocean_month_' + clim_str + '.clim.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_clim_snapshot = xr.open_dataset(pp_diags_folder + 'ocean_snapshot_month_' + clim_str + '.clim.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).load()

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_clim = ds_clim.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim.time.values]})
ds_clim_snapshot = ds_clim_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_clim_snapshot.time.values]})

# Add wrap-around for clim_snapshot:
tminus1 = str(ds_clim_snapshot.time.isel(time=-1).values)
tminus1 = np.datetime64(str(int(tminus1[:4])-1) + tminus1[4:])
ds_clim_snapshot = xr.concat([ds_clim_snapshot.isel(time=-1).assign_coords({'time':tminus1}),ds_clim_snapshot],dim='time')

### Load snapshots for plotting

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

# Analyse a specific event/region

This section contains various plotting features to analyse an event of interest.

First, define the region of interest (for spatial averaging) and its name:

In [ ]:
sreg = [150-360,168-360,-44,-38] # Tasman Sea region from Kajtar et al. 2022
region_name = 'Tasman Sea'
#sreg = [-100, 20, 0, 60] # North Atlantic region used in England et al. 2025
#region_name = 'North Atlantic'

Then, compute a daily-resolution climatology covering the period of interest

In [ ]:
add_extra = True # Whether to add a second year in the climatologies in order to cover the second half of December

year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
year_clim = int(str(ds_clim.time[0].astype('datetime64[Y]').values)[:4])

# climatology, standard variables:
ds_climA = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in ds_clim.time]})
if add_extra:
    ds_clim2 = ds_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in ds_clim.time]})
    ds_climA = xr.concat([ds_climA,ds_clim2],dim='time')
ds_climA_daily = ds_climA.resample(time='1D').interpolate("linear").sel(time=times)

# Climatology, budget variables:
mlt_budget_stavg_climA = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in mlt_budget_stavg_clim.time]})
if add_extra:
    mlt_budget_stavg_clim2 = mlt_budget_stavg_clim.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in mlt_budget_stavg_clim.time]})
    mlt_budget_stavg_climA = xr.concat([mlt_budget_stavg_climA,mlt_budget_stavg_clim2],dim='time')

## Plot time series of mixed layer temperature, mixed layer depth and budget terms averaged over the event region

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=1,figsize=(12,16),height_ratios=[1.,0.5,1.,1.])

# Panel 1: Mixed layer temperature, including climatology and snapshots:
mlt = (ds_day.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt_snap = (ds_day_snapshot.temp_in_mld/rho0).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
mlt.plot(ax=axes[0],label='Daily-average mixed layer temperature',linewidth=5.)
mlt_snap.plot(ax=axes[0],label='Snapshot mixed layer temperature',linewidth=2.,linestyle='dashed')
(ds_climA_daily.temp_in_mld/rho0).plot(ax=axes[0],label=clim_label + ' climatology',linewidth=2.)

# # Plot some averages:
# Octavg = mlt.sel(time=slice('2017-10-01','2017-10-31')).mean('time')
# Decavg = mlt.sel(time=slice('2017-12-01','2017-12-31')).mean('time')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-11-01')],[Octavg.values,Octavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-12-01'),np.datetime64('2018-01-01')],[Decavg.values,Decavg.values],'-',color='C0',linewidth=2.)
# axes[0].plot([np.datetime64('2017-10-16T12:00:00'),np.datetime64('2017-12-16T12:00:00')],[Octavg.values,Decavg.values],'-',color='C0',linewidth=2.,linestyle='dashed',label='Epoch difference (Dec minus Oct)')
# axes[0].plot([np.datetime64('2017-10-01'),np.datetime64('2017-12-31')],[mlt_snap.sel(time='2017-10-01',method='nearest'),mlt_snap.sel(time='2017-12-31',method='nearest')],'-',color='C1',linewidth=2.,linestyle='dotted',label='Snapshot difference (24Z 31st Dec - 24Z 1st Oct)')

axes[0].legend()
axes[0].set_title(region_name + ' mixed layer temperature budget')
axes[0].set_ylabel('Temperature ($\circ$C)')
axes[0].grid()

# Panel 2: Mixed layer depth and climatology:
ds_day.mld.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean']).plot(ax=axes[1],linewidth=2.,label='Mixed layer depth')
(ds_climA_daily.mld).plot(ax=axes[1],label=clim_label + ' climatology',linewidth=2.)
axes[1].set_ylabel('Mixed layer depth (m)')
axes[1].grid()
axes[1].legend()
axes[1].set_ylim([0.,150.])

# Panel 3: Budget terms (raw)
vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux','sw_pen']
labels = ['Tendency','Entrainment','Advection','Vertical Mixing','Surface fluxes','SW penetration']

budget_stavg = mlt_budget_stavg_daily.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])
budget_stavg_clim = mlt_budget_stavg_climA
unit_conv = 86400

for j, var in enumerate(vars):
    if j == 0:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j] + ' (st. avg.)')
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1.,label=labels[j] + ' (' + clim_label + ' climat.)')
    else:
        (budget_stavg[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=2.,label=labels[j])
        (budget_stavg_clim[var]*unit_conv).plot(ax=axes[2],color='C' + str(j),linewidth=1)
axes[2].legend()
axes[2].set_ylabel('Temperature tendency ($\circ$C/day)')
axes[2].grid()

# Panel 4: Budget terms (anomalies):

budget_stavg_clim_daily = mlt_budget_stavg_climA.interp(time=budget_stavg.time.astype('datetime64[s]'))
for j, var in enumerate(vars):
    ((budget_stavg[var]-budget_stavg_clim_daily[var])*unit_conv).plot(ax=axes[3],color='C' + str(j),linewidth=2,label=labels[j])
axes[3].legend()
axes[3].set_ylabel('Anomalous temperature \n tendency ($\circ$C/day)')
axes[3].grid()

for ax in axes:
    ax.set_xlabel('')
    ax.set_xlim([mlt.time[0],mlt.time[-1]])
    
#plt.savefig('MLT_budget_' + region_name.replace(' ','') + '_time_series_with_anomalies.png',dpi=250,bbox_inches='tight')

## Plot spatial plot of mixed layer temperature anomalies during event

Currently this is just for the Tasman Sea 2017 event

In [ ]:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(12,3.6),layout='constrained')

mlt = (ds_day.temp_in_mld.resample(time='1ME').mean()/rho0)
mlt_clim = (ds_clim.temp_in_mld/rho0)

(mlt.sel(time='2017-10').drop_vars(['time'])-mlt_clim.sel(time='1989-10').drop_vars(['time'])).plot(ax=axes[0],add_colorbar=False)
(mlt.sel(time='2017-11').drop_vars(['time'])-mlt_clim.sel(time='1989-11').drop_vars(['time'])).plot(ax=axes[1],add_colorbar=False)
(mlt.sel(time='2017-12').drop_vars(['time'])-mlt_clim.sel(time='1989-12').drop_vars(['time'])).plot(ax=axes[2],cbar_kwargs={'label': 'Mixed layer temperature \n anomaly ($^\circ$C)'})

axes[0].set_title('October 2017')
axes[1].set_title('November 2017')
axes[2].set_title('December 2017')
for ax in axes:
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
axes[1].set_yticklabels([])
axes[2].set_yticklabels([])

#plt.savefig('MLT_budget_TasmanSea_OctDec2017_MLTA.png',dpi=300,bbox_inches='tight')

## Spatial plots of monthly time-averaged budget anomalies

In [ ]:
# Time integrate daily budget to get monthly budget (units are degrees celsius change over that month, so degC/month accounting for the different lengths of months):
mlt_budget_stavg_monthly = (mlt_budget_stavg_daily*(ds_day.average_DT/np.timedelta64(1,'s'))).resample(time='1ME').sum()
mlt_budget_stavg_monthly = mlt_budget_stavg_monthly.assign_coords({'time':mlt_budget_stavg_monthly.time.astype('datetime64[M]')}) # Replace time stamp with year-month only (to match climatology)

# Compute budget anomalies by subtracting climatology:
year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])                # Year of interest
mon_len = (ds_day.average_DT/np.timedelta64(1,'s')).resample(time='1ME').sum()    # Length of month in seconds   
mon_len = mon_len.assign_coords({'time':mon_len.time.astype('datetime64[M]')})    
anomaly_budget = mlt_budget_stavg_monthly - mlt_budget_stavg_clim.assign_coords({'time':[np.datetime64(str(year) + '-01')+np.timedelta64(x,'M') for x in range(12)]})*mon_len # Subtract climatology (times length of month in seconds to get same units)

# Compute spatial averages for each month over focus region (for bar plots further down):
anomaly_budget_av = (anomaly_budget*ds_grid.area_t).sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])/ds_grid.area_t.sel(xt_ocean=slice(sreg[0],sreg[1]),yt_ocean=slice(sreg[2],sreg[3])).mean(['xt_ocean','yt_ocean'])

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12,6))
axs = axes.reshape(-1)

#times = ['2023-06','2023-07']
#time_labels = ['June 2023','July 2023']
#times = ['2023-05','2023-06']
#time_labels = ['May 2023','June 2023']
times = ['2017-10','2017-11']
time_labels = ['Oct 2017','Nov 2017']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['advection','vert_mixing','entrainment']]
names = ['MLT tendency','Net surface flux term','Advection, mixing and entrainment']
clims = [-2.4,2.4]
unit_conv = 1.

for i,time in enumerate(times):
    budget_ti = anomaly_budget.sel(time=np.datetime64(time,'M'))*unit_conv

    for j, varg in enumerate(vars):
        ds = budget_ti[varg[0]]
        if len(varg)>1:
            for k in range(len(varg)-1):
                ds += budget_ti[varg[k+1]]
        ds.plot.contourf(ax=axes[i][j],levels=np.arange(-2.4,2.8,0.4),cmap='RdBu_r')
        axes[i][j].set_title(names[j] + ' (' + time_labels[i] + ')')
        axes[i][j].set_xlabel('')
        axes[i][j].set_ylabel('')
        axes[i][j].set_facecolor([0.5,0.5,0.5])
        axes[i][j].set_xlim(reg[:2])
        axes[i][j].set_ylim([reg[2],reg[3]])

axes[0][0].plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')
    
plt.tight_layout()
#plt.savefig('MLT_budget_NorthAtlantic_spatial_May_Jun.png',dpi=200,bbox_inches='tight')

## Bar plots of region averages

This uses `anomaly_budget_av` computed above

In [ ]:
fig = plt.figure(figsize=(12,8))
ax = plt.gca()

#times = ['2023-05','2023-06','2023-07','2023-08']
#time_labels = ['May','June','July','August']
times = ['2017-09','2017-10','2017-11','2017-12']
time_labels = ['Sep 2017','Oct 2017','Nov 2017','Dec 2017']
vars = [['mlt_tendency'],
        ['surface_flux','sw_pen'],
        ['shortwave','sw_pen'],
        ['latent'],
        ['longwave'],
        ['sensible'],
        ['vert_mixing','entrainment'],
       ['advection']]
names = ['MLT tendency','Net surface flux','Shortwave','Latent','Longwave','Sensible','Vertical mixing and entrainment','Advection']

# Create variable groups:
budget_av_gr = anomaly_budget_av['mlt_tendency'].rename(names[0]).to_dataset()
for i, varg in enumerate(vars[1:]):
    budget_av_gr[names[i+1]] = anomaly_budget_av[varg[0]]
    if len(varg)>1:
        for k in range(len(varg)-1):
            budget_av_gr[names[i+1]] += anomaly_budget_av[varg[k+1]]
    
df = budget_av_gr.rename({'time':'class'}).assign_coords({'class':time_labels}).to_dataframe().reset_index()
   
# Parameters
num_vars = len(times)
num_classes = len(names)
bar_width = 0.1
x = np.arange(num_vars)  # One x position per variable

unit_conv = 1.
# Create figure

# Plot each class as a separate bar group
for i, cls in enumerate(names):
    # Get values for this class across all variables
    values = list(df[cls]*unit_conv)
    
    # Offset x positions for each class
    ax.bar(x + i * bar_width, values, width=bar_width, label=cls)

# Formatting
ax.set_xticks(x + bar_width)
ax.set_xticklabels(time_labels)
#ax.set_yticks(np.arange(-0.8,3.2,0.4))
#ax.set_ylim([-0.8,1.8])
ax.set_ylabel("$^\circ$C/month")
#ax.set_title('North Atlantic MLT budget anomalies 2023')
ax.set_title('MLT budget anomalies')
ax.legend(title="Budget terms",fontsize=8,loc='upper right',bbox_to_anchor=[1.1,0.9,0.1,0.1])
ax.grid()
plt.tight_layout()
#plt.savefig('MLT_budget_NorthAtlantic_time_series_big.png',dpi=200,bbox_inches='tight')

## Spatial plots of time-averaged budgets comparing standard and hat-averaging

In this section we demonstrate the use of hat-averaging by comparing budgets averaged over different time periods.

First, define whether we want to work with anomalies or full field budgets/variables:

In [ ]:
# Here we subselect variables and times for efficiency:
vars = ['advection','vert_mixing','surface_flux','sw_pen','fixedh_tendency']
vars_st = vars + ['mlt_tendency','entrainment']
times = slice(None,None)
stavg_daily = mlt_budget_stavg_daily[vars_st].isel(time=times)
falavg_daily = mlt_budget_falavg_daily[vars].isel(time=times)
mlt_daily = (ds_day.temp_in_mld/rho0).isel(time=times).load()
average_DT = ds_day.average_DT.isel(time=times).load()

In [ ]:
# If you want budget anomalies, run this cell:

# Compute spatially-resolved climatologies:
# ---------------------------
add_extra = True # Whether to add a second year in the climatologies in order to cover the second half of December

year = int(str(ds_day.time[0].astype('datetime64[Y]').values)[:4])
year_clim = int(str(ds_clim.time[0].astype('datetime64[Y]').values)[:4])

# climatology, standard variables:
ds_climA = ds_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in ds_clim.time]})
if add_extra:
    ds_clim2 = ds_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in ds_clim.time]})
    ds_climA = xr.concat([ds_climA,ds_clim2],dim='time')

# Climatology, budget variables:
mlt_budget_stavg_climA = mlt_budget_stavg_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in mlt_budget_stavg_clim.time]})
if add_extra:
    mlt_budget_stavg_clim2 = mlt_budget_stavg_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in mlt_budget_stavg_clim.time]})
    mlt_budget_stavg_climA = xr.concat([mlt_budget_stavg_climA,mlt_budget_stavg_clim2],dim='time')

mlt_budget_falavg_climA = mlt_budget_falavg_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year))) for x in mlt_budget_falavg_clim.time]})
if add_extra:
    mlt_budget_falavg_clim2 = mlt_budget_falavg_clim.assign_coords({'time':[np.datetime64(str(x.values).replace(str(year_clim),str(year+1))) for x in mlt_budget_falavg_clim.time]})
    mlt_budget_falavg_climA = xr.concat([mlt_budget_falavg_climA,mlt_budget_falavg_clim2],dim='time')

# Subtract them from the full field values to produce anomalies:
stavg_daily -= mlt_budget_stavg_climA[vars_st].interp(time=stavg_daily.time.astype('datetime64[s]'))
falavg_daily -= mlt_budget_falavg_climA[vars].interp(time=falavg_daily.time.astype('datetime64[s]'))
mlt_daily -= (ds_climA.temp_in_mld/rho0).interp(time=mlt_daily.time.astype('datetime64[s]')).load()

Now compute the standard average and rising average daily *difference* budgets (i.e the standard average here corresponds to the temperature difference between the beginning and ending of the day in degC, rather than a temperature tendency in degC/sec)

In [ ]:
stavg_daily_diff = stavg_daily*(average_DT/np.timedelta64(1,'s'))
falavg_daily_diff = falavg_daily*(average_DT/np.timedelta64(1,'s'))
risavg_daily_diff = stavg_daily_diff - falavg_daily_diff

In [ ]:
# # Quick check plot:
# var = 'advection'
# fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(12,8))
# clim = 0.1
# stavg_daily_diff[var].isel(time=0).plot(ax=axes[0][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
# axes[0][0].set_title('Standard average')
# risavg_daily_diff[var].isel(time=0).plot(ax=axes[1][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
# axes[1][0].set_title('Rising average')
# falavg_daily_diff[var].isel(time=0).plot(ax=axes[0][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
# axes[0][1].set_title('Falling average')
# (risavg_daily_diff - falavg_daily_diff)[var].isel(time=0).plot(ax=axes[1][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
# axes[1][1].set_title('Rising - Falling')
# plt.tight_layout()

# This looks correct (testing with advection): Falling and rising averages look similar to the standard average but multipled by a factor of about 0.5 
# (the area under the curve of a diagonal line from 0 to 1). The difference between the rising and the falling average
# is smaller again, and corresponds to changes in the budget term of the course of the day.

Next, we compute monthly difference budgets.

For standard averaging this is just a sum operation summing up the differences for each day

In [ ]:
# Monthly standard average difference budget:
stavg_monthly_diff = stavg_daily_diff.resample(time='1ME').sum()

For rising averages this is more difficult, as it requires a cumulative integral to account for the fact that the weighting for the rising (or falling) averages begins again at zero for each diagnostic accumulation period (here, daily). The maths explaining the next cell is contained at the bottom of the `Theory_and_Diagnostics.ipynb` notebook.

In [ ]:
# Define n-1 DataArrays for the months:
n_minus_1 = xr.DataArray(data=[x-1 for x in risavg_daily_diff.time.dt.day.values],dims=['time'],coords={'time':stavg_daily_diff.time})

# Monthly rising average:
risavg_monthly_diff = risavg_daily_diff.resample(time='1ME').mean() + (stavg_daily_diff*n_minus_1).resample(time='1ME').mean()

Finally, we can now compute the hat average for the budget difference between any two months of interest. This is done by a function defined in the next cell (which should work for either daily or monthly hat differences, providing you provide the right inputs).

In [ ]:
# Define function to compute hat-averaged budget difference between two indices of interest
def hat_difference(risavg_diff_budget,stavg_diff_budget,mlt,t1_index,t2_index):
    """
    Compute hat difference mlt budget from time index 1 to index 2. This function should work
    for either days or months.

    Inputs:
    - risavg_diff_budget: The difference budget, rising average
    - stavg_diff_budget: The difference budget, standard average
    - mlt: The time-averaged (over either days or months) mixed layer tracer
    - t1_index: The first time index
    - t2_index: The second time index
    """

    # List of variables:
    vars = list(risavg_diff_budget.data_vars)

    # Interim months:
    tM_index = np.arange(t1_index+1,t2_index,1)
    
    # Compute mlt difference as "mlt_tendency":
    hatavg_diff_budget = (mlt.isel(time=t2_index) - mlt.isel(time=t1_index)).rename('mlt_tendency').to_dataset()

    # Compute falling difference as difference between other budgets:
    falavg_diff_budget = stavg_diff_budget - risavg_diff_budget

    # Compute budget terms:
    for var in vars:
        hatavg_diff_budget[var] = risavg_diff_budget[var].isel(time=t1_index) + stavg_diff_budget[var].isel(time=tM_index).sum('time') + falavg_diff_budget[var].isel(time=t2_index)

    # Compute entrainment by residual:
    hatavg_diff_budget['entrainment'] = -(hatavg_diff_budget['fixedh_tendency'] - hatavg_diff_budget['mlt_tendency'])

    return(hatavg_diff_budget)

In [ ]:
# Choose months of interest:
t1_index = 1
t2_index = 3

# Compute hatavg difference (i.e. the budget contributions to the difference between the time-averaged MLT in month 2 and month 1):
hatavg_monthly_diff = hat_difference(risavg_monthly_diff,
                                     stavg_monthly_diff,
                                     mlt_daily.resample(time='1ME').mean(),
                                     t1_index,t2_index)

# Compute standard average difference (i.e. the budget contributions to the difference between the MLT snapshots at the end of month 2 to the start of month 1)
stavg_monthly_diff = stavg_monthly_diff.isel(time=slice(t1_index,t2_index+1)).sum('time')

In [ ]:
# Compute surface flux absorbed:
stavg_monthly_diff['surface_flux_absorbed'] = stavg_monthly_diff['surface_flux'] + stavg_monthly_diff['sw_pen']
hatavg_monthly_diff['surface_flux_absorbed'] = hatavg_monthly_diff['surface_flux'] + hatavg_monthly_diff['sw_pen']

In [ ]:
fig, axs = plt.subplots(nrows=2, ncols=5, figsize=(12,5.5),layout='constrained')

vars = ['mlt_tendency','entrainment','advection','vert_mixing','surface_flux_absorbed']
labels = ['MLT difference','Entrainment','Advection','Vertical Mixing','Surface fluxes']
budgets = [stavg_monthly_diff.where(~np.isnan(mlt_daily.isel(time=0))),
           hatavg_monthly_diff]
budget_labels = ['Snapshot difference\n24Z 31st Dec - 24Z 30th Sep, 2017',
                 'Epoch difference\nDec - Oct, 2017']
unit_conv = 1
clims = [10,10,10,20,20]
fname = 'MLT_budget_TasmanSea_Hat_St_Avg_Comparison.png'

#clims = [4,4,4,8,8]
#fname = 'MLT_budget_TasmanSea_Hat_St_Avg_Comparison_Anomalies.png'

for i in range(len(budgets)):
    for j, var in enumerate(vars):
        if i == 0:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',cbar_kwargs={'label':labels[j] + ' ($^\circ$C)','shrink':0.7,'location':'top','ticks':[-clims[j],0,clims[j]]})
        else:
            (budgets[i][var]*unit_conv).plot(ax=axs[i][j],cmap='RdBu_r',vmin=-clims[j],vmax=clims[j],extend='both',add_colorbar=False)
        axs[i][j].set_ylabel('')
    axs[i][0].set_ylabel(budget_labels[i])

for i, ax in enumerate(axs.reshape(-1)):
    ax.set_xlabel('')
    ax.set_title('')
    ax.set_facecolor('k')
    ax.set_yticks([-60,-50,-40,-30])
    ax.set_xticks([-220,-210,-200,-190])
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.text(-224,-21,f'({string.ascii_lowercase[i]})',verticalalignment='top',color='w')
    ax.set_facecolor([0.5,0.5,0.5])
    ax.plot([sreg[0],sreg[1],sreg[1],sreg[0],sreg[0]],[sreg[2],sreg[2],sreg[3],sreg[3],sreg[2]],'-k')

axs[0][0].set_yticklabels(['60S','50S','40S','30S'])
axs[1][0].set_yticklabels(['60S','50S','40S','30S'])
for ax in axs[1]:
    ax.set_xticklabels(['220W','210W','200W','190W'])
    
plt.savefig(fname,dpi=250,bbox_inches='tight')